# Pathology terms and patient risk (post-landmark text)

Standalone exploratory analysis of the platinum-trained, transferred XGBoost score at ADT +0 / +90 / +180 days.
Run manuscript notebook 04 first. This notebook reads its `transfer_scores.csv` and the pathology parquet directly;
it does **not** read Elio's extreme-risk tail manifest or create patient exports. All finite scored patients are eligible,
including the middle of the risk distribution.

The [Elio data request / high vs low risk patient extraction](../../patient_extraction/01_high_vs_low_risk_transfer_pxs.ipynb)
is separate. Aggregate analysis outputs go to `<COMPASS data root>/testing/pathology_topics_vs_risk/`.

TF-IDF compares the highest-risk 10% with the lowest-risk 10% at each landmark, using group means,
mean differences and a two-sided Mann-Whitney test of patient weights. Binary term enrichment, the curated lexicon,
and binary term/lexicon associations with risk percentile remain available. BERTopic analyses are in
[15_pathology_bertopic_vs_risk.ipynb](15_pathology_bertopic_vs_risk.ipynb).

Outputs include `pathology_tfidf_high_vs_low.csv` and `pathology_tfidf_group_coverage.csv`, plus
`pathology_term_enrichment.csv`, `pathology_lexicon_enrichment.csv`, and `pathology_score_association.csv`.
Only aggregate tables are saved; no patient IDs or raw report text.

**Timing: post-landmark text only.** At each landmark, a report is used when ADT + landmark < report date ≤
ADT + `WINDOW_MAX_DAYS` (3650 d, the 10-year cap of notebook 11). Days run from ADT start
(`TREATMENT_ANCHOR_DATE` in `DATA_PATH`) to the report's `EVENT_DATE`, dated as in notebook 11. Undated and
pre-landmark reports are dropped. The window is applied per landmark, so a report at ADT + 120 d is used at +0 and +90
but not at +180. Outputs go to the `post_landmark/` subfolder of the output folder.

These are descriptive associations with risk, not prospective validation or causal effects. Post-landmark pathology
may reflect progression/treatment; they are not what was on file when the patient was scored.

For pooled and per-note-type analyses of Clinician, Imaging, and Pathology notes, use
[14_clinical_notes_topics_vs_risk.ipynb](14_clinical_notes_topics_vs_risk.ipynb).


## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
PATHOLOGY_PATH = Path(os.environ.get(
    "PROFILE_PATHOLOGY_NOTES",
    "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/CLINICAL_NOTES/PATHOLOGY_NOTES.parquet"))
OUTPUT_DIR = DATA_ROOT / "text_interpretation"

# Post-landmark text only: at each landmark, notes with ADT + landmark < EVENT_DATE <= ADT + WINDOW_MAX_DAYS.
# Undated notes are dropped. Outputs go to the post_landmark/ subfolder.
WINDOW_MAX_DAYS = 3650             # same 10-year cap as notebook 11
DATA_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"   # ADT anchor dates (TREATMENT_ANCHOR_DATE)
OUTPUT_DIR = OUTPUT_DIR / "post_landmark"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (0, 90, 180)
RISK_TAIL_FRACTION = 0.10           # highest 10% versus lowest 10% at each landmark

# Pathology term enrichment (top vs bottom risk tail)
TERM_MIN_PATIENTS = 5              # a term must appear in >= this many patients at the landmark
TERM_NGRAMS = (1, 2)               # unigrams and bigrams
TERM_FDR = 0.05                    # Benjamini-Hochberg threshold for the displayed tables
TERM_TOP_N = 25                    # terms shown per landmark x group

# Stratified sensitivity tests (Mantel-Haenszel, van Elteren): report-count band x any non-prostate specimen
ADJUST_REPORT_BANDS = (1, 2, 4)    # band lower edges: 1, 2-3, 4+ reports
SPECIMEN_HEADER_CHARS = 300        # report characters (after PROC_DESC) searched for the specimen site

# Direct high-versus-low TF-IDF comparison (no resampling).
TFIDF_SETTINGS = dict(min_group_patients=10, block_size=256)


## Risk scores (all patients)

One full-model XGBoost score per patient and landmark. Invalid scores are excluded. Decile labels support the legacy
contrasts only; coincident cutoffs leave all finite scores in the middle group and do not block continuous analyses.


In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def risk_tail_flags(g):
    if not 0 < RISK_TAIL_FRACTION < 0.5:
        raise ValueError("RISK_TAIL_FRACTION must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    lo, hi = risk[valid].quantile([RISK_TAIL_FRACTION, 1 - RISK_TAIL_FRACTION]) if valid.any() else (np.nan, np.nan)
    if not valid.any() or lo >= hi:
        print(f"Landmark {landmark}: no distinct risk-tail cutoffs; legacy extreme-group contrasts unavailable.")
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    # Only the two risk tails are labelled; everything in between stays blank.
    g["decile_group"] = pd.Series(pd.NA, index=g.index, dtype="string")
    if lo < hi:
        g.loc[risk >= hi, "decile_group"] = "top"
        g.loc[risk <= lo, "decile_group"] = "bottom"
    g.loc[~valid, "decile_group"] = "not scored"
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: risk_tail_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

## Pathology term enrichment: top vs bottom risk tail

At each landmark, the patients in the **highest 10%** and **lowest 10%** of that landmark's score are compared (ties at a
cutoff stay in the extreme groups; patients in neither risk tail are excluded from these group comparisons). Two contrasts per landmark, each reading
"overrepresented in `group` relative to `control`": top vs bottom and bottom vs top (both directions, because the test
is one-sided). Each patient's group can differ between landmarks.

- **Reports:** the post-landmark pathology reports of each scored patient (see the overview). Groups come from each
  landmark's score. Post-landmark reports include post-progression pathology, so terms can reflect the outcome the
  model predicts rather than what was on file when the patient was scored. Patients with no
  pathology report in the window drop out of that landmark's denominators.
- **Terms:** lowercased letter-only unigrams and bigrams (numbers, dates and IDs drop out), English stop words removed,
  counted within each report so bigrams never span two reports. A term is kept if it appears in at least
  `TERM_MIN_PATIENTS` patients at the landmark, which reduces rare strings but does not guarantee de-identification; keep outputs protected.
- **Test:** a patient "has" a term if any of their reports contains it. For each contrast, one-sided Fisher's
  exact test for overrepresentation in `group`, Benjamini-Hochberg q-values within each contrast, and the odds ratio
  with a +0.5 correction. Mean patient-level TF-IDF (sublinear term frequency over each patient's pooled counts) is reported
  for each group as a descriptive column. The `*_mh` columns repeat the test stratified by report count and specimen
  site (Mantel-Haenszel; see below).

In [ ]:
import re

import polars as pl

TERM_GROUPS = ("top", "bottom")
CONTROL_GROUP = "middle"   # patients in neither decile: a label only, never analysed
# (group, control): one-sided tests for terms / topics overrepresented in group relative to control.
CONTRASTS = (("top", "bottom"), ("bottom", "top"))   # one-sided, so both directions

# --- Specimen site (for the stratified sensitivity test): the site named first in PROC_DESC + the report's opening.
SPECIMEN_SITES = (
    ("prostate", re.compile(r"\b(?:prostat\w*|turp|transurethral|seminal\s+vesicles?)\b")),
    ("non-prostate", re.compile(r"\b(?:bone|marrow|liver|hepatic|lymph|nodes?|nodal|lung|pulmonary|pleura\w*|adrenal"
                                r"|brain|dura\w*|spine|spinal|vertebra\w*|soft\s+tissue|peritone\w*|omentum|omental)\b")),
)


def specimen_site(header):
    """'prostate', 'non-prostate' or 'unspecified': whichever site is named first in a lowercased header."""
    hits = [(m.start(), site) for site, rx in SPECIMEN_SITES for m in [rx.search(header)] if m]
    return min(hits)[1] if hits else "unspecified"


# Per-landmark group for every scored patient; a blank decile group means scored but in neither decile.
groups_by_lm = {}
for lm, g in by_lm.items():
    ids = pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64"), name="DFCI_MRN")
    grp = pd.Series(g["decile_group"].fillna(CONTROL_GROUP).to_numpy(), index=ids, name="group")
    groups_by_lm[lm] = grp[grp.ne("not scored")]

scored_ids = sorted({int(i) for grp in groups_by_lm.values() for i in grp.index.dropna()})
# RPT_ID (when present) links reports to the optional embeddings; PROC_DESC (when present) helps type the specimen.
note_cols = ["DFCI_MRN", "EVENT_DATE", "RPT_TEXT"]
schema = pl.scan_parquet(PATHOLOGY_PATH).collect_schema().names()
note_cols += [c for c in ("RPT_ID", "PROC_DESC") if c in schema]
all_notes = (pl.scan_parquet(PATHOLOGY_PATH)
             .select(note_cols)
             .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids))
             .collect().to_pandas())
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes["note_row"] = np.arange(len(all_notes))      # row position, used to look up per-report lexicon hits

# Specimen site, once per report.
header = all_notes["RPT_TEXT"].str[:SPECIMEN_HEADER_CHARS]
if "PROC_DESC" in all_notes:
    header = all_notes["PROC_DESC"].fillna("").astype(str) + " " + header
all_notes["specimen_site"] = header.str.lower().map(specimen_site) if len(all_notes) else pd.Series(dtype=object)


# --- Post-landmark window. Days from ADT start = EVENT_DATE (its YYYY-MM-DD prefix) - TREATMENT_ANCHOR_DATE,
# dated as in notebook 11. Undated notes are dropped.
anchor_cols = ["DFCI_MRN", "TREATMENT_ANCHOR_DATE"]
anchor = (pd.read_parquet(DATA_PATH, columns=anchor_cols) if str(DATA_PATH).endswith((".parquet", ".pq"))
          else pd.read_csv(DATA_PATH, usecols=anchor_cols, low_memory=False)).dropna().drop_duplicates()
anchor["TREATMENT_ANCHOR_DATE"] = pd.to_datetime(anchor["TREATMENT_ANCHOR_DATE"], errors="coerce", format="mixed")
anchor = anchor.dropna().drop_duplicates()
anchor["DFCI_MRN"] = pd.to_numeric(anchor["DFCI_MRN"], errors="raise").astype("int64")
if anchor["DFCI_MRN"].duplicated().any():
    raise ValueError("More than one TREATMENT_ANCHOR_DATE for some patients.")
anchor = anchor.set_index("DFCI_MRN")["TREATMENT_ANCHOR_DATE"]
n_missing = sum(i not in anchor.index for i in scored_ids)
if n_missing:
    raise ValueError(f"{n_missing} scored patients lack an ADT anchor in DATA_PATH; reconcile it with the scores.")
note_date = pd.to_datetime(all_notes["EVENT_DATE"].astype("string").str.extract(r"^(\d{4}-\d{2}-\d{2})", expand=False),
                           errors="coerce")
all_notes["days_from_adt"] = (note_date - all_notes["DFCI_MRN"].map(anchor)).dt.days.astype(float).to_numpy()


def in_window(reports, lm):
    """Rows of `reports` after ADT + `lm` and at or before ADT + WINDOW_MAX_DAYS (undated rows never)."""
    days = reports["days_from_adt"]
    return (days.gt(lm) & days.le(WINDOW_MAX_DAYS)).fillna(False).to_numpy(dtype=bool)


def window_summary(rows):
    """Prints, per landmark, how many of the scored patients' notes are post-landmark (aggregate counts only)."""
    print(f"Post-landmark notes (ADT + landmark < note date <= ADT + {WINDOW_MAX_DAYS} d; undated notes dropped):")
    for r in rows:
        print(f"  landmark {r['landmark']}: {r['in_window']:,} of {r['any_date']:,} notes "
              f"({r['undated']:,} undated), {r['patients_in_window']:,} patients")


# The post-landmark reports of the patients scored at each landmark.
# Each patient's stratum: report count band (ADJUST_REPORT_BANDS) x any non-prostate-site report, within the window.
reports_by_lm, strata_by_lm, size_rows, window_rows = {}, {}, [], []
report_cols = ["DFCI_MRN", "RPT_TEXT", "specimen_site", "note_row"] + note_cols[3:]
for lm, grp in groups_by_lm.items():
    pool = all_notes[all_notes["DFCI_MRN"].isin(grp.index)]
    rep_lm = pool.loc[in_window(pool, lm), report_cols].reset_index(drop=True)
    reports_by_lm[lm] = rep_lm
    window_rows.append(dict(landmark=lm, any_date=len(pool), in_window=len(rep_lm),
                            undated=int(pool["days_from_adt"].isna().sum()),
                            patients_in_window=int(rep_lm["DFCI_MRN"].nunique())))
    per = (rep_lm.assign(non_prostate=rep_lm["specimen_site"].eq("non-prostate"))
           .groupby("DFCI_MRN").agg(reports=("non_prostate", "size"), non_prostate=("non_prostate", "any")))
    band = np.searchsorted(ADJUST_REPORT_BANDS, per["reports"].to_numpy(), side="right") - 1
    strata_by_lm[lm] = pd.Series(band * 2 + per["non_prostate"].to_numpy(dtype=int), index=per.index, name="stratum")
    has_report = grp.index.isin(rep_lm["DFCI_MRN"])
    for name in TERM_GROUPS:
        in_group = grp.eq(name).to_numpy()
        members = per.index.isin(grp.index[in_group])
        size_rows.append({"landmark": lm, "group": name, "patients_scored": int(in_group.sum()),
                          "patients_with_report": int((in_group & has_report).sum()),
                          "reports": int(rep_lm["DFCI_MRN"].isin(grp.index[in_group]).sum()),
                          "median_reports": float(per.loc[members, "reports"].median()) if members.any() else np.nan,
                          "patients_with_non_prostate_report": int(per.loc[members, "non_prostate"].sum())})

sites = all_notes["specimen_site"].value_counts()
print(f"Specimen site of the {len(all_notes):,} pathology reports of scored patients (all dates): "
      + ", ".join(f"{site} {int(sites.get(site, 0)):,}" for site in ("prostate", "non-prostate", "unspecified")))
window_summary(window_rows)
group_sizes = pd.DataFrame(size_rows).set_index(["landmark", "group"])
group_sizes


In [ ]:
from scipy import sparse
from scipy.stats import hypergeom, norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# Applied after lowercasing: letters only, >= 3 characters.
TOKEN_PATTERN = r"(?u)\b[a-z][a-z]{2,}\b"
MH_COLS = ["odds_ratio_mh", "p_value_mh", "q_value_mh"]
ENRICHMENT_COLS = ["landmark", "group", "control", "term", "n_group", "n_control", "group_with_term", "control_with_term",
                   "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS,
                   "mean_tfidf_group", "mean_tfidf_control"]


def bh_qvalues(p):
    """Benjamini-Hochberg q-values over the finite p values; NaN stays NaN (cluster scipy predates
    false_discovery_control)."""
    p = np.asarray(p, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if ok.size:
        order = ok[np.argsort(p[ok])]
        ranked = p[order] * ok.size / np.arange(1, ok.size + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def patient_term_matrices(reports, ids):
    """Patient x term presence and TF-IDF, with terms counted within each report and summed per patient.
    """
    text = reports["RPT_TEXT"].to_numpy()
    vec = CountVectorizer(lowercase=True, stop_words="english", token_pattern=TOKEN_PATTERN,
                          ngram_range=TERM_NGRAMS, min_df=TERM_MIN_PATIENTS)
    try:
        # Report-level min_df is a safe prefilter: a term in >= k patients is in >= k reports.
        counts = vec.fit_transform(text)
    except ValueError:  # no term survives (or fewer reports than TERM_MIN_PATIENTS)
        return np.array([], dtype=object), None, None
    rows = ids.get_indexer(reports["DFCI_MRN"])
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))),
                                   shape=(len(ids), len(rows)))
    patient_counts = (to_patient @ counts).tocsc()
    present = (patient_counts > 0).astype(np.int64)
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return np.array([], dtype=object), None, None
    patient_counts, present = patient_counts[:, keep], present[:, keep]
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(patient_counts)
    return vec.get_feature_names_out()[keep], present.tocsr(), tfidf.tocsr()


def colsum(m):
    return np.asarray(m.sum(axis=0)).ravel().astype(float)


def mantel_haenszel(present, in_group, control, strata):
    """One-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio per feature column, across strata.

    A stratum without patients from both sides carries no information and is skipped; if none is left, both are NaN.
    The odds ratio adds 0.5 to every cell of a stratum that has an empty cell.
    """
    n_feat = present.shape[1]
    A, E, V, R, S = (np.zeros(n_feat) for _ in range(5))
    informative = False
    for k in pd.unique(strata[in_group | control]):
        g, c = in_group & (strata == k), control & (strata == k)
        n1, n0 = int(g.sum()), int(c.sum())
        if not (n1 and n0):
            continue
        informative = True
        n = n1 + n0
        a, cc = colsum(present[g]), colsum(present[c])
        b, d, m = n1 - a, n0 - cc, a + cc
        A += a
        E += n1 * m / n
        V += n1 * n0 * m * (n - m) / (n ** 2 * (n - 1))
        h = 0.5 * ((a == 0) | (b == 0) | (cc == 0) | (d == 0))
        R += (a + h) * (d + h) / (n + 4 * h)
        S += (b + h) * (cc + h) / (n + 4 * h)
    if not informative:
        return np.full(n_feat, np.nan), np.full(n_feat, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(V > 0, norm.sf((A - E - 0.5) / np.sqrt(V)), 1.0)
    return R / S, p


def fisher_enrichment(present, label, features, landmark, strata=None):
    """One-sided Fisher overrepresentation of each feature column for each (group, control) in CONTRASTS.

    present: patients x features 0/1 sparse matrix; label: each patient's group; strata (optional): each patient's
    stratum for the Mantel-Haenszel columns (NaN without it). Rows come out in CONTRASTS order, len(features) rows
    per contrast whose group and control both have patients.
    """
    out = []
    for name, ref in CONTRASTS:
        in_group, control = label == name, label == ref
        n1, n0 = int(in_group.sum()), int(control.sum())
        if n1 == 0 or n0 == 0:
            continue
        a, c = colsum(present[in_group]), colsum(present[control])
        p = hypergeom.sf(a - 1, n1 + n0, a + c, n1)       # P(X >= a): one-sided Fisher
        if strata is None:
            or_mh = p_mh = np.full(len(features), np.nan)
        else:
            or_mh, p_mh = mantel_haenszel(present, in_group, control, np.asarray(strata))
        out.append(pd.DataFrame({
            "landmark": landmark, "group": name, "control": ref, "feature": features, "n_group": n1, "n_control": n0,
            "group_with_feature": a.astype(int), "control_with_feature": c.astype(int),
            "pct_group": a / n1, "pct_control": c / n0,
            "odds_ratio": ((a + 0.5) * (n0 - c + 0.5)) / ((n1 - a + 0.5) * (c + 0.5)),
            "p_value": p, "q_value": bh_qvalues(p),
            "odds_ratio_mh": or_mh, "p_value_mh": p_mh, "q_value_mh": bh_qvalues(p_mh),
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def sort_contrasts(df):
    """Landmark, then CONTRASTS order, then p value and odds ratio."""
    rank = {pair: i for i, pair in enumerate(CONTRASTS)}
    order = pd.Series([rank[pair] for pair in zip(df["group"], df["control"])], index=df.index, dtype=float)
    return (df.assign(_contrast=order)
            .sort_values(["landmark", "_contrast", "p_value", "odds_ratio"], ascending=[True, True, True, False])
            .drop(columns="_contrast").reset_index(drop=True))


def contrast_title(lm, name, ref):
    return f"Landmark {lm} · {name} {RISK_TAIL_FRACTION:.0%} vs {ref} {RISK_TAIL_FRACTION:.0%}"


term_matrices = {}   # landmark -> (patient ids, terms, presence, TF-IDF); reused by the score association cell


def term_enrichment(reports, groups, landmark):
    """Fisher overrepresentation of each term for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(reports["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 report
    terms, present, tfidf = patient_term_matrices(reports, ids)
    term_matrices[landmark] = (ids, terms, present, tfidf)
    if not len(terms):
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    label = groups.reindex(ids).to_numpy()
    res = fisher_enrichment(present, label, terms, landmark, strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    mean_tfidf = {name: np.asarray(tfidf[label == name].mean(axis=0)).ravel()
                  for name in set(res["group"]) | set(res["control"])}
    pairs = list(res[["group", "control"]].drop_duplicates().itertuples(index=False))
    res["mean_tfidf_group"] = np.concatenate([mean_tfidf[g] for g, _ in pairs])
    res["mean_tfidf_control"] = np.concatenate([mean_tfidf[c] for _, c in pairs])
    return res.rename(columns={"feature": "term", "group_with_feature": "group_with_term",
                               "control_with_feature": "control_with_term"})


enrichment = pd.concat(
    [term_enrichment(reports_by_lm[lm], groups_by_lm[lm], lm) for lm in LANDMARKS if lm in reports_by_lm],
    ignore_index=True,
).reindex(columns=ENRICHMENT_COLS)
enrichment = sort_contrasts(enrichment)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
enrichment_path = OUTPUT_DIR / "pathology_term_enrichment.csv"
enrichment.to_csv(enrichment_path, index=False)
print(f"wrote {enrichment_path} ({len(enrichment):,} term x contrast rows)")


## TF-IDF: high versus low risk

Compare the highest-risk 10% with the lowest-risk 10% at each landmark. `RISK_TAIL_FRACTION` controls both tails;
ties at a cutoff stay together, so actual group sizes can exceed 10%. Patients with no report in the selected time
window do not enter the comparison. The shared vocabulary and IDF are fitted across eligible scored patients at
that landmark, as in the term-enrichment analysis; only high- and low-risk patients enter the TF-IDF group test.

Each patient contributes one normalized TF-IDF vector, including zero weights for absent terms. The table reports
mean TF-IDF in each group, high-minus-low mean difference, and `auc_high_vs_low`: the probability that a high-risk
patient has a greater weight than a low-risk patient, with ties counting half. An AUC of 0.5 indicates no directional
advantage. Mean differences and AUCs describe different aspects of the distributions and can disagree in direction.

A two-sided Mann-Whitney test compares the weight distributions, with tie and continuity corrections and an
asymptotic p-value. Require at least `TFIDF_SETTINGS['min_group_patients']` patients in each group for inference;
smaller groups retain descriptive summaries with missing p/q-values. BH correction spans all tested terms and
landmarks in this notebook/timing folder. Both directions share one two-sided test. There are no permutations,
bootstrap loops, continuous-risk correlations, or adjusted regressions in this TF-IDF analysis.

Weights describe relative textual prominence, not absolute mention counts or disease severity. Outputs are
`pathology_tfidf_high_vs_low.csv` and `pathology_tfidf_group_coverage.csv`. The coverage table includes the configured
tail fraction, score cutoffs and patient counts. Only aggregate results are saved.


In [ ]:
from COMPASS.multivariable_models import text_tfidf_association as tfa

try:
    from IPython.display import display
except ImportError:
    display = print

tfidf_frames, tfidf_coverage = [], []
for lm in LANDMARKS:
    entry = term_matrices.get(lm)
    groups = groups_by_lm[lm]
    report_ids = reports_by_lm[lm]["DFCI_MRN"].unique()
    tfidf_coverage.append(dict(landmark=lm, tail_fraction=RISK_TAIL_FRACTION,
        high_cutoff=by_lm[lm].attrs["hi"], low_cutoff=by_lm[lm].attrs["lo"],
        high_scored=int(groups.eq("top").sum()), low_scored=int(groups.eq("bottom").sum()),
        high_with_report=int((groups.eq("top") & groups.index.isin(report_ids)).sum()),
        low_with_report=int((groups.eq("bottom") & groups.index.isin(report_ids)).sum()),
        n_terms=0 if entry is None else len(entry[1])))
    if entry is None or not len(entry[1]):
        continue
    ids, terms, _, tfidf = entry
    tfidf_frames.append(tfa.compare_tfidf_groups(tfidf, ids, terms, groups, lm, **TFIDF_SETTINGS))

tfidf_comparison = tfa.adjust_fdr(tfidf_frames)
tfidf_comparison.to_csv(OUTPUT_DIR / "pathology_tfidf_high_vs_low.csv", index=False)
pd.DataFrame(tfidf_coverage).to_csv(OUTPUT_DIR / "pathology_tfidf_group_coverage.csv", index=False)
print(f"TF-IDF: highest {RISK_TAIL_FRACTION:.0%} vs lowest {RISK_TAIL_FRACTION:.0%}; "
      f"{len(tfidf_comparison):,} term x landmark rows")
display(pd.DataFrame(tfidf_coverage))
print(f"{tfidf_comparison['status'].eq('ok').sum():,} tested; "
      f"{tfidf_comparison['q_value'].lt(TERM_FDR).sum():,} at q < {TERM_FDR} across terms and landmarks")
for lm in LANDMARKS:
    sub = tfidf_comparison[tfidf_comparison["landmark"].eq(lm)]
    for direction, mask in (("high", sub["difference_high_minus_low"].gt(0)),
                            ("low", sub["difference_high_minus_low"].lt(0))):
        print(f"Landmark {lm}: terms with greater mean TF-IDF in the {direction}-risk group")
        display(sub[mask].sort_values("p_value")[
            ["term", "n_high", "n_low", "mean_tfidf_high", "mean_tfidf_low", "difference_high_minus_low",
             "auc_high_vs_low", "p_value", "q_value", "status"]].head(TERM_TOP_N).round(4).reset_index(drop=True))
if tfidf_comparison.empty:
    print("No retained vocabulary to compare.")
elif tfidf_comparison['status'].ne('ok').any():
    display(tfidf_comparison.loc[tfidf_comparison['status'].ne('ok'), 'status'].value_counts().rename("skipped_terms"))


In [ ]:
try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

show_cols = ["term", "group_with_term", "n_group", "pct_group", "control_with_term", "n_control", "pct_control",
             "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh", "mean_tfidf_group", "mean_tfidf_control"]
for lm in LANDMARKS:
    for name, ref in CONTRASTS:
        sub = enrichment[(enrichment["landmark"] == lm) & (enrichment["group"] == name) & (enrichment["control"] == ref)]
        title = contrast_title(lm, name, ref)
        if sub.empty:
            print(f"\n{title}: no terms (too few patients or reports)")
            continue
        sig = sub[sub["q_value"] < TERM_FDR]
        note = f"q < {TERM_FDR}: {len(sig):,} terms"
        if sig.empty:
            sig, note = sub, f"none at q < {TERM_FDR}; top {TERM_TOP_N} by p shown"
        top = sig.sort_values(["p_value", "odds_ratio"], ascending=[True, False]).head(TERM_TOP_N)
        print(f"\n{title} ({note})")
        display(top[show_cols].round(3).reset_index(drop=True))

## Sensitivity and complementary analyses

Same groups, landmarks and post-landmark reports as the term enrichment. Each one addresses a specific way the term
tables can mislead.

1. **Report count and specimen type (`*_mh` columns).** Patients with more reports have more chances to mention any
   term, and metastatic-site biopsies read differently from prostate specimens. Each patient's stratum is their
   report-count band (`ADJUST_REPORT_BANDS`) crossed with whether any report is from a
   non-prostate site (from `PROC_DESC` and the report's first `SPECIMEN_HEADER_CHARS` characters). The term and lexicon
   tables (and the topic tables in notebook 15) add a one-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio across strata.
   These are NaN when no stratum contains patients from both sides of the contrast.
2. **Curated lexicon (`pathology_lexicon_enrichment.csv`).** The neuroendocrine patterns are copied from the NEPC
   annotation pipeline's `ne_gate.py`, joined by individual NE and prostate IHC markers, histologic variants, invasion
   / extension terms and the highest Gleason sum (grade group mapped to a sum). Same Fisher and MH tests.
3. **Continuous risk (`pathology_score_association.csv`).** Deciles discard most of the score. For every term and
   lexicon feature, a two-sided Mann-Whitney test of the risk percentile among all scored patients with a
   report, feature present vs absent. `auc` = P(a patient with the feature ranks higher than one without). The van
   Elteren test (`*_adj`) stratifies it like item 1.

All outputs are aggregate. No MRN or report text is printed or written by these cells.

In [ ]:
# Curated lexicon, matched on lowercased report text. The NE patterns are copied verbatim (not imported) from
# LLM_clinical_annotations/tasks/nepc/ne_gate.py (NE_GATE_VERSION "ne-gate-v3", commit
# edb1713ef454f6ece313fa675b5f9438937f831e); update them by hand if that file changes.
NE_TERM = (r"\b(?:neuro[\s-]?endocrine|t-?nepc|nepc|small[\s-]?cell|scpc|scnc|"
           r"oat[\s-]?cell)\b")
SMALL_CELL_TERM = r"\b(?:small[\s-]?cell|scpc|scnc|oat[\s-]?cell)\b"
NE_DIAGNOSIS = (r"\b(?:t-?nepc|nepc|scpc|scnc)\b|"
                r"\b(?:neuro[\s-]?endocrine|small[\s-]?cell|oat[\s-]?cell)"
                r"(?:\s+[\w/()-]+){0,4}?\s+"
                r"(?:carcinoma|cancer|tumou?r|neoplasm|malignancy|component|histology|"
                r"variant|subtype|type|disease|prostate)\b|"
                r"\b(?:carcinoma|cancer)\s+(?:with|of)\s+(?:\w+\s+){0,2}?"
                r"(?:neuro[\s-]?endocrine|small[\s-]?cell)\b")
NE_MARKER = (r"\b(?:synaptophysin|chromogranin(?:\s+a)?|cd\s?56|insm-?1|"
             r"neuron[\s-]+specific\s+enolase|nse)\b")
NON_PROSTATE_NE = (r"\b(?:sclc|small[\s-]?cell\s+lung|merkel|carcinoid|"
                   r"pulmonary\s+(?:origin|primary|small)|lung\s+(?:origin|primary)|"
                   r"urothelial|bladder\s+(?:primary|origin|cancer|carcinoma|tumou?r)|"
                   r"pancreatic\s+(?:net|neuroendocrine)|pnet|gep[\s-]?nets?|"
                   r"well[\s-]differentiated\s+neuroendocrine\s+tumou?r)\b")
NON_SMALL_CELL = r"\bnon[\s-]*$"
# (end of the ne_gate.py copy)

LEXICON = {
    # neuroendocrine (ne_gate.py), then each NE marker from NE_MARKER on its own
    "NE term (any)": NE_TERM,
    "NE diagnosis phrase": NE_DIAGNOSIS,
    "small cell": SMALL_CELL_TERM,
    "NE marker (any)": NE_MARKER,
    "non-prostate NE primary": NON_PROSTATE_NE,
    "synaptophysin": r"\bsynaptophysin\b",
    "chromogranin": r"\bchromogranin\b",
    "CD56": r"\bcd\s?56\b",
    "INSM1": r"\binsm-?1\b",
    "NSE": r"\b(?:neuron[\s-]+specific\s+enolase|nse)\b",
    # other IHC
    "Ki-67 / MIB-1": r"\b(?:ki-?67|mib-?1)\b",
    "AR": r"\b(?:androgen\s+receptor|ar)\b",
    "NKX3.1": r"\bnkx\s?3[.\s-]?1\b",
    "PSMA": r"\b(?:psma|prostate[\s-]specific\s+membrane\s+antigen)\b",
    "ERG": r"\berg\b",
    "PTEN": r"\bpten\b",
    # histology and staging
    "ductal": r"\bductal\b",
    "intraductal": r"\bintraductal\b|\bidc-?p\b",
    "cribriform": r"\bcribriform\b",
    "sarcomatoid": r"\bsarcomatoid\b",
    "treatment effect": r"\b(?:treatment|therapy|hormon\w*|androgen[\s-]deprivation)[\s-]+(?:related\s+)?effects?\b",
    "perineural invasion": r"\bperineural\s+(?:invasion|infiltration)\b|\bpni\b",
    "lymphovascular invasion": (r"\b(?:lympho-?vascular|angiolymphatic|lymphatic|vascular)\s+(?:space\s+)?invasion\b"
                                r"|\blvi\b"),
    "extraprostatic extension": r"\bextra-?prostatic\s+extension\b|\bepe\b",
    "seminal vesicle invasion": r"\bseminal\s+vesicles?\s+(?:invasion|involvement|involved)\b|\bsvi\b",
}
LEXICON_RE = {name: re.compile(rx) for name, rx in LEXICON.items()}
NON_PREFIX_RE = re.compile(NON_SMALL_CELL)
# Gleason: the highest primary + secondary sum in any report (grade group mapped to a sum).
GLEASON_RE = re.compile(r"\bgleason\b[^.;\n]{0,40}?\b([1-5])\s*\+\s*([1-5])\b")
GRADE_GROUP_RE = re.compile(r"\b(?:grade\s+group|isup\s+grade(?:\s+group)?|gg)\s*:?\s*([1-5])\b")
GRADE_GROUP_SUM = {1: 6, 2: 7, 3: 7, 4: 8, 5: 9}
GLEASON_BINS = (("Gleason max 2-6", 2, 6), ("Gleason max 7", 7, 7), ("Gleason max 8-10", 8, 10))

MENTION_FEATURES = list(LEXICON)
LEXICON_FEATURES = MENTION_FEATURES + [label for label, _, _ in GLEASON_BINS]
LEXICON_COLS = ["landmark", "group", "control", "feature", "n_group", "n_control", "group_with_feature",
                "control_with_feature", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def lexicon_hits(text):
    """One report's lexicon features: {feature: True} for each mention, plus gleason_max.

    A match right after "non-" ("non-small cell") is not a mention.
    """
    text = text.lower()
    hits = {}
    for name, rx in LEXICON_RE.items():
        if any(not NON_PREFIX_RE.search(text[max(0, m.start() - 8):m.start()]) for m in rx.finditer(text)):
            hits[name] = True
    sums = ([int(a) + int(b) for a, b in GLEASON_RE.findall(text)]
            + [GRADE_GROUP_SUM[int(g)] for g in GRADE_GROUP_RE.findall(text)])
    if sums:
        hits["gleason_max"] = max(sums)
    return hits


# Report-level hits, once for every report at any landmark.
report_rows = np.unique(np.concatenate([rep["note_row"].to_numpy() for rep in reports_by_lm.values()]
                                       + [np.array([], dtype=int)])).astype(int)
report_lex = pd.DataFrame([lexicon_hits(t) for t in all_notes["RPT_TEXT"].to_numpy()[report_rows]],
                          index=report_rows).reindex(columns=MENTION_FEATURES + ["gleason_max"])
report_mentions = report_lex[MENTION_FEATURES].notna().to_numpy()
report_gleason = pd.to_numeric(report_lex["gleason_max"], errors="coerce").to_numpy(dtype=float)
lexicon_matrices = {}   # landmark -> (patient ids, presence); reused by the score association cell
lexicon_frames = []
for lm in LANDMARKS:
    if lm not in reports_by_lm or reports_by_lm[lm].empty:
        continue
    rep = reports_by_lm[lm]
    ids = pd.Index(sorted(rep["DFCI_MRN"].unique()), dtype="Int64")
    rows = ids.get_indexer(rep["DFCI_MRN"])
    pos = np.searchsorted(report_rows, rep["note_row"].to_numpy())
    mentions = np.zeros((len(ids), len(MENTION_FEATURES)), dtype=bool)
    np.logical_or.at(mentions, rows, report_mentions[pos])
    gleason = np.full(len(ids), np.nan)
    np.fmax.at(gleason, rows, report_gleason[pos])
    bins = np.column_stack([(gleason >= lo) & (gleason <= hi) for _, lo, hi in GLEASON_BINS])
    present = sparse.csr_matrix(np.hstack([mentions, bins]).astype(np.int64))
    lexicon_matrices[lm] = (ids, present)
    # Only features some patient has at this landmark are tested, so absent ones do not dilute the BH correction.
    seen = np.flatnonzero(colsum(present) > 0)
    lexicon_frames.append(fisher_enrichment(present[:, seen], groups_by_lm[lm].reindex(ids).to_numpy(),
                                            np.array(LEXICON_FEATURES, dtype=object)[seen], lm,
                                            strata_by_lm[lm].reindex(ids).to_numpy()))

# Rows stay in landmark, CONTRASTS, then LEXICON_FEATURES order.
lexicon_enrich = (pd.concat(lexicon_frames, ignore_index=True) if lexicon_frames else pd.DataFrame()
                  ).reindex(columns=LEXICON_COLS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lexicon_path = OUTPUT_DIR / "pathology_lexicon_enrichment.csv"
lexicon_enrich.to_csv(lexicon_path, index=False)
print(f"wrote {lexicon_path} ({len(lexicon_enrich):,} feature x contrast rows)")

# One wide table per landmark: % of patients with each feature by group, then q values for every contrast
# (q = Fisher, q_mh = stratified Mantel-Haenszel). Features no patient has at that landmark are not tested.
for lm in LANDMARKS:
    sub = lexicon_enrich[lexicon_enrich["landmark"] == lm]
    if sub.empty:
        print(f"\nLandmark {lm} · lexicon: no patients with pathology reports")
        continue
    wide = pd.DataFrame(index=pd.Index(LEXICON_FEATURES, name="feature"))
    for name, ref in CONTRASTS:
        c = sub[(sub["group"] == name) & (sub["control"] == ref)].set_index("feature")
        if c.empty:
            continue
        wide[f"% {name}"], wide[f"% {ref}"] = c["pct_group"] * 100, c["pct_control"] * 100
        wide[f"q {name}>{ref}"], wide[f"q_mh {name}>{ref}"] = c["q_value"], c["q_value_mh"]
    pct_cols = [f"% {g}" for g in TERM_GROUPS if f"% {g}" in wide]
    q_cols = [col for prefix in ("q ", "q_mh ") for col in wide.columns if col.startswith(prefix)]
    wide = wide[pct_cols + q_cols]
    print(f"\nLandmark {lm} · curated lexicon (% of patients with a mention in any report)")
    display(wide.dropna(how="all").round(3))


In [ ]:
from scipy.stats import rankdata

ASSOC_COLS = ["landmark", "feature_set", "feature", "n_patients", "n_with_feature", "pct_with_feature",
              "mean_percentile_with", "mean_percentile_without", "auc", "p_value", "q_value",
              "auc_adj", "p_value_adj", "q_value_adj"]


def _mann_whitney_parts(present, score):
    """Per feature column: n with it, U of the with-feature patients, and the tie-corrected null variance of U."""
    n = present.shape[0]
    ranks = rankdata(score)
    n1 = colsum(present)
    u = np.asarray(present.T @ ranks).ravel() - n1 * (n1 + 1) / 2
    _, t = np.unique(score, return_counts=True)
    ties = float((t.astype(float) ** 3 - t).sum()) / (n * (n - 1)) if n > 1 else 0.0
    var = n1 * (n - n1) / 12 * ((n + 1) - ties)
    return n1, u, var


def score_association(present, features, score, strata, landmark, feature_set):
    """Two-sided Mann-Whitney of the risk percentile, feature present vs absent, among all scored patients with
    pathology reports; and the van Elteren test stratified like the Mantel-Haenszel columns.

    auc = P(percentile of a patient with the feature > one without); auc_adj pools the same comparison within strata.
    """
    present = present.tocsr().astype(float)
    n = present.shape[0]
    n1, u, var = _mann_whitney_parts(present, score)
    n0 = n - n1
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.maximum(np.abs(u - n1 * n0 / 2) - 0.5, 0) / np.sqrt(var)
        p = np.where(var > 0, 2 * norm.sf(z), np.nan)
        sum_with = np.asarray(present.T @ score).ravel()
        res = pd.DataFrame({
            "landmark": landmark, "feature_set": feature_set, "feature": features, "n_patients": n,
            "n_with_feature": n1.astype(int), "pct_with_feature": n1 / n,
            "mean_percentile_with": sum_with / n1, "mean_percentile_without": (score.sum() - sum_with) / n0,
            "auc": u / (n1 * n0), "p_value": p})
    # van Elteren: within-stratum U, strata weighted 1 / (N_k + 1).
    num, den, u_pool, pairs = (np.zeros(len(features)) for _ in range(4))
    for k in pd.unique(strata):
        rows = np.flatnonzero(strata == k)
        if rows.size < 2:
            continue
        n1k, uk, vk = _mann_whitney_parts(present[rows], score[rows])
        w = 1 / (rows.size + 1)
        num += w * (uk - n1k * (rows.size - n1k) / 2)
        den += w ** 2 * vk
        u_pool += uk
        pairs += n1k * (rows.size - n1k)
    with np.errstate(divide="ignore", invalid="ignore"):
        res["auc_adj"] = np.where(pairs > 0, u_pool / pairs, np.nan)
        res["p_value_adj"] = np.where(den > 0, 2 * norm.sf(np.abs(num) / np.sqrt(den)), np.nan)
    res = res[(res["n_with_feature"] > 0) & (res["n_with_feature"] < n)].reset_index(drop=True)
    res["q_value"], res["q_value_adj"] = bh_qvalues(res["p_value"]), bh_qvalues(res["p_value_adj"])
    return res[ASSOC_COLS]


assoc_frames = []
for lm in LANDMARKS:
    if lm not in by_lm:
        continue
    g = by_lm[lm]
    pct = pd.Series(g["risk_percentile"].to_numpy(dtype=float),
                    index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64")))
    sets = []
    if lm in term_matrices and len(term_matrices[lm][1]):
        ids, terms, present, _ = term_matrices[lm]
        sets.append(("term", ids, present, terms))
    if lm in lexicon_matrices:
        ids, present = lexicon_matrices[lm]
        sets.append(("lexicon", ids, present, np.array(LEXICON_FEATURES, dtype=object)))
    for feature_set, ids, present, names in sets:
        if len(ids) < 2:
            continue
        assoc_frames.append(score_association(present, names, pct.reindex(ids).to_numpy(dtype=float),
                                              strata_by_lm[lm].reindex(ids).to_numpy(), lm, feature_set))

assoc = (pd.concat(assoc_frames, ignore_index=True) if assoc_frames else pd.DataFrame()).reindex(columns=ASSOC_COLS)
assoc_path = OUTPUT_DIR / "pathology_score_association.csv"
assoc.to_csv(assoc_path, index=False)
print(f"wrote {assoc_path} ({len(assoc):,} feature x landmark rows)")

assoc_show = ["feature", "n_with_feature", "pct_with_feature", "mean_percentile_with", "mean_percentile_without",
              "auc", "q_value", "auc_adj", "q_value_adj"]
for lm in LANDMARKS:
    for feature_set in ("term", "lexicon"):
        sub = assoc[(assoc["landmark"] == lm) & (assoc["feature_set"] == feature_set)]
        if sub.empty:
            print(f"\nLandmark {lm} · {feature_set}s vs risk percentile: none tested")
            continue
        for direction, side in (("higher", sub["auc"] > 0.5), ("lower", sub["auc"] < 0.5)):
            hits = sub[side].sort_values("p_value")
            print(f"\nLandmark {lm} · {feature_set}s with {direction} risk percentiles "
                  f"(q < {TERM_FDR}: {int((hits['q_value'] < TERM_FDR).sum()):,}; top {TERM_TOP_N} by p)")
            display(hits[assoc_show].head(TERM_TOP_N).round(3).reset_index(drop=True))
